**Before running:** Runtime → Change runtime type → **T4 GPU**. Run cell 1, then cell 2 (about 5–8 min; it downloads `final.zip` at the end).

Open this notebook fresh from GitHub (File → Open notebook → GitHub → `246mayank/cuda-sgemm`) or upload it from your PC, not from Recent.

In [ ]:
# 1 — confirm the GPU
!nvidia-smi

In [ ]:
# 2 — COMPLETE BENCHMARK (about 5-8 min): build, v0-v9, tensor cores, tuning, bandwidth, SASS, Nsight Compute, clock lock
%cd /content
!rm -rf cuda-sgemm && git clone -q https://github.com/246mayank/cuda-sgemm
%cd /content/cuda-sgemm
!git log --oneline -1
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
import datetime
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d_%H%M%S')
print('== build'); get_ipython().system('make 2>&1 | grep -E "error|spill" | head -40; ls -la bench bandwidth')
print('== 1/9 benchmark v0-v9'); get_ipython().system('./bench')
print('== 2/9 tensor-core track'); get_ipython().system('./bench --tc')
print('== 3/9 tuning study'); get_ipython().system('./bench --tune')
print('== 4/9 bandwidth'); get_ipython().system('./bandwidth')
print('== 5/9 SASS')
!cuobjdump -sass bench | gzip > results/sass_{stamp}.txt.gz && ls -la results/sass_{stamp}.txt.gz
print('== 6/9 ncu v0-v9 at N=2048')
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_dram_{stamp}.csv --metrics dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "A exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_mem_{stamp}.csv --metrics l1tex__t_sector_hit_rate.pct,lts__t_sector_hit_rate.pct,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,l1tex__t_requests_pipe_lsu_mem_global_op_ld.sum,sm__warps_active.avg.pct_of_peak_sustained_active ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "B exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v[0-9] --log-file results/ncu_smem_{stamp}.csv --metrics l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum,l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_st.sum,smsp__inst_executed_op_shared_ld.sum,smsp__sass_thread_inst_executed_op_ffma_pred_on.sum,smsp__inst_executed.sum ./bench --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "C exit: $?"
print('== 7/9 ncu tensor cores at N=2048')
!ncu --csv --print-units base -k regex:tc[0-9]_ --log-file results/ncu_tcdram_{stamp}.csv --metrics dram__bytes_read.sum,dram__bytes_write.sum,gpu__time_duration.sum ./bench --tc --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "T1 exit: $?"
!ncu --csv --print-units base -k regex:tc[0-9]_ --log-file results/ncu_tcpipe_{stamp}.csv --metrics sm__pipe_tensor_cycles_active.avg.pct_of_peak_sustained_active,sm__inst_executed_pipe_tensor.sum ./bench --tc --size 2048 --warmup 0 --reps 1 --no-write > /dev/null; echo "T2 exit: $?"
print('== 8/9 v0 1023 vs 1024')
!ncu --csv --print-units base -k regex:sgemm_v0 --log-file results/ncu_v0stride_1023_{stamp}.csv --metrics dram__bytes_read.sum,l1tex__t_sector_hit_rate.pct,lts__t_sector_hit_rate.pct,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,gpu__time_duration.sum ./bench --kernel v0 --size 1023 --warmup 0 --reps 1 --no-write > /dev/null; echo "D 1023 exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v0 --log-file results/ncu_v0set_1023_{stamp}.csv --metrics l1tex__t_set_accesses_pipe_lsu_mem_global_op_ld.sum,l1tex__t_set_conflicts_pipe_lsu_mem_global_op_ld.sum ./bench --kernel v0 --size 1023 --warmup 0 --reps 1 --no-write > /dev/null; echo "E 1023 exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v0 --log-file results/ncu_v0stride_1024_{stamp}.csv --metrics dram__bytes_read.sum,l1tex__t_sector_hit_rate.pct,lts__t_sector_hit_rate.pct,l1tex__t_sectors_pipe_lsu_mem_global_op_ld.sum,gpu__time_duration.sum ./bench --kernel v0 --size 1024 --warmup 0 --reps 1 --no-write > /dev/null; echo "D 1024 exit: $?"
!ncu --csv --print-units base -k regex:sgemm_v0 --log-file results/ncu_v0set_1024_{stamp}.csv --metrics l1tex__t_set_accesses_pipe_lsu_mem_global_op_ld.sum,l1tex__t_set_conflicts_pipe_lsu_mem_global_op_ld.sum ./bench --kernel v0 --size 1024 --warmup 0 --reps 1 --no-write > /dev/null; echo "E 1024 exit: $?"
print('== 9/9 clock-locked run')
!nvidia-smi -lgc 900,900 > results/clock_lock_{stamp}.txt 2>&1; echo "lgc exit: $?" >> results/clock_lock_{stamp}.txt; cat results/clock_lock_{stamp}.txt
!grep -q 'lgc exit: 0' results/clock_lock_{stamp}.txt && ./bench --prefix locked900_ > /dev/null; nvidia-smi -rgc > /dev/null 2>&1; echo "locked run done"
!zip -qr final.zip results
from google.colab import files; files.download("final.zip")

Kaggle fallback: same cells; download results.zip from the Output panel instead of cell 3. Use `make ARCH=sm_60` if the GPU is a P100 (ncu metric names and clock locking may differ).